# SurgBox: dataset and detectors

Loads the SurgBox annotations and the two detectors from Hugging Face, and compares their boxes on test frames.

```bash
pip install datasets ultralytics huggingface_hub opencv-python matplotlib pandas
```

The Cholec80 videos are not included. Request them from [CAMMA](http://camma.u-strasbg.fr/datasets) and set `CHOLEC80` to the folder with `video01.mp4` ... `video80.mp4`.

In [ ]:
import cv2
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import pandas as pd
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from ultralytics import RTDETR, YOLO

CHOLEC80 = "cholec80/videos"  # folder with video01.mp4 ... video80.mp4

surgbox = load_dataset("M-Hamdy/SurgBox")
CLASSES = surgbox["train"].features["objects"]["category"].feature.names
surgbox

## Dataset

In [ ]:
rows = []
for split, d in surgbox.items():
    cats = [c for o in d["objects"] for c in o["category"]]
    rows.append({"split": split, "videos": len(set(d["video"])), "frames": len(d), "boxes": len(cats),
                 **{name: cats.count(i) for i, name in enumerate(CLASSES)}})
pd.DataFrame(rows).set_index("split")

In [ ]:
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7"]

def read_frame(video, frame):
    path = f"{CHOLEC80}/video{video:02d}.mp4"
    cap = cv2.VideoCapture(path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame)
    ok, image = cap.read()
    if not ok:
        raise FileNotFoundError(f"cannot read frame {frame} of {path}; set CHOLEC80 to the folder with the Cholec80 videos")
    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

def show(ax, image, boxes, title):
    ax.imshow(image)
    ax.set_title(title, fontsize=10)
    ax.axis("off")
    for (x, y, w, h), c in boxes:
        ax.add_patch(patches.Rectangle((x, y), w, h, fill=False, lw=2, color=COLORS[c]))
        ax.text(x + 3, y + 14, CLASSES[c], color="white", fontsize=7, backgroundcolor=COLORS[c])

# test frames (unseen by the detectors) with the most instrument types, one per video
test = surgbox["test"].to_pandas()
test["n_classes"] = test["objects"].map(lambda o: len(set(o["category"])))
frames = test.sort_values(["n_classes", "video", "frame"], ascending=[False, True, True]).drop_duplicates("video").head(3)

fig, axes = plt.subplots(1, 3, figsize=(15, 3.2))
for ax, r in zip(axes, frames.itertuples()):
    show(ax, read_frame(r.video, r.frame), list(zip(r.objects["bbox"], r.objects["category"])), f"video {r.video}, frame {r.frame}")
plt.tight_layout()
plt.show()

## Detectors

In [ ]:
models = {
    "YOLOv8-x": YOLO(hf_hub_download("M-Hamdy/SurgBox-YOLOv8-x", "yolov8x_surgbox.pt")),
    "RT-DETR-L": RTDETR(hf_hub_download("M-Hamdy/SurgBox-RT-DETR-L", "rtdetr-l_surgbox.pt")),
}

def detect(model, image):
    boxes = model.predict(image[..., ::-1], imgsz=640, conf=0.25, verbose=False)[0].boxes
    return [((x1, y1, x2 - x1, y2 - y1), int(c)) for (x1, y1, x2, y2), c in zip(boxes.xyxy.tolist(), boxes.cls.tolist())]

fig, axes = plt.subplots(len(frames), 3, figsize=(15, 3.2 * len(frames)))
for row, r in zip(axes, frames.itertuples()):
    image = read_frame(r.video, r.frame)
    show(row[0], image, list(zip(r.objects["bbox"], r.objects["category"])), f"SurgBox, video {r.video}")
    for ax, (name, model) in zip(row[1:], models.items()):
        show(ax, image, detect(model, image), name)
plt.tight_layout()
plt.show()